# LiteEdit-Qwen — 02: Step Count Sweep

**Goal:** Sweep diffusion steps 5 → 50 across all three tasks and plot quality vs speed.

**Key finding to look for:** Background replacement tolerates low steps better than local editing.

**Outputs:**
- `results/figures/step_sweep.png` — quality/speed curves (Figure 2 in paper)
- `results/metrics/step_sweep.csv` — raw numbers

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import copy, time
import torch
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from paths import PATHS
from models.loader import load_model
from tasks.bg_replace import BgReplaceTask
from tasks.person_remove import PersonRemoveTask
from tasks.local_edit import LocalEditTask
from eval.metrics import MetricsTracker

PATHS.figures.mkdir(parents=True, exist_ok=True)
PATHS.metrics.mkdir(parents=True, exist_ok=True)
print('Setup complete.')

In [ ]:
# ── Load INT4 model once ──────────────────────────────────────────
model, processor, base_cfg = load_model('configs/quant_int4.yaml')
tracker = MetricsTracker(device='cuda')
print('Model ready.')

In [ ]:
# ── Sweep ─────────────────────────────────────────────────────────
STEP_COUNTS = [5, 10, 20, 30, 50]

TASK_CONFIGS = {
    'bg_replace': {
        'cls':    BgReplaceTask,
        'input':  'data/test/bg_replace/input/img_0000.png',
        'ref':    'data/test/bg_replace/ref/img_0000.png',
        'prompt': 'Replace the background with a snowy mountain landscape',
        'mask':   None,
    },
    'person_remove': {
        'cls':    PersonRemoveTask,
        'input':  'data/test/person_remove/input/img_0000.png',
        'ref':    'data/test/person_remove/ref/img_0000.png',
        'prompt': 'Remove the person from the scene',
        'mask':   'data/test/person_remove/mask/img_0000.png',
    },
    'local_edit': {
        'cls':    LocalEditTask,
        'input':  'data/test/local_edit/input/img_0000.png',
        'ref':    'data/test/local_edit/ref/img_0000.png',
        'prompt': 'Change the color of the highlighted region to blue',
        'mask':   'data/test/local_edit/mask/img_0000.png',
    },
}

records = []

for task_name, tc in TASK_CONFIGS.items():
    in_img   = Image.open(tc['input']).convert('RGB')
    ref_img  = Image.open(tc['ref']).convert('RGB')
    mask_img = Image.open(tc['mask']).convert('L') if tc['mask'] else None

    print(f'\nTask: {task_name}')
    for steps in STEP_COUNTS:
        cfg = copy.deepcopy(base_cfg)
        cfg['num_steps'] = steps

        task = tc['cls'](model, processor, cfg)
        tracker.start_timer()
        out, _ = task.run(image=in_img, prompt=tc['prompt'], mask=mask_img)
        latency, vram = tracker.stop_timer()

        m = tracker.compute(pred=out, ref=ref_img, original=in_img, mask=mask_img)

        row = {'task': task_name, 'steps': steps,
               'lpips': round(m['lpips'], 4), 'psnr': round(m['psnr'], 2),
               'latency_s': round(latency, 3), 'peak_vram_gb': round(vram, 3)}
        if 'outside_psnr' in m:
            row['outside_psnr'] = round(m['outside_psnr'], 2)
        records.append(row)
        print(f'  steps={steps:3d}  LPIPS={row["lpips"]:.4f}  '
              f'PSNR={row["psnr"]:.2f}  lat={latency:.2f}s')

df = pd.DataFrame(records)
csv_path = PATHS.metrics / 'step_sweep.csv'
df.to_csv(csv_path, index=False)
print(f'\nCSV saved → {csv_path}')

In [ ]:
# ── Plot and save to results/figures/ ─────────────────────────────
COLORS = {'bg_replace': '#4878CF', 'person_remove': '#D65F5F', 'local_edit': '#3DA35D'}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for task_name in TASK_CONFIGS:
    sub = df[df['task'] == task_name].sort_values('steps')
    c = COLORS[task_name]
    axes[0].plot(sub['steps'], sub['lpips'],     'o-', color=c, label=task_name)
    axes[1].plot(sub['steps'], sub['latency_s'], 'o-', color=c, label=task_name)
    axes[2].plot(sub['latency_s'], sub['lpips'], 'o-', color=c, label=task_name)
    for _, row in sub.iterrows():
        axes[2].annotate(str(int(row['steps'])),
                         xy=(row['latency_s'], row['lpips']),
                         xytext=(4, 2), textcoords='offset points', fontsize=8)

axes[0].set_xlabel('Diffusion steps'); axes[0].set_ylabel('LPIPS ↓'); axes[0].set_title('Quality vs steps')
axes[1].set_xlabel('Diffusion steps'); axes[1].set_ylabel('Latency (s)'); axes[1].set_title('Speed vs steps')
axes[2].set_xlabel('Latency (s)'); axes[2].set_ylabel('LPIPS ↓'); axes[2].set_title('Quality–speed trade-off')

for ax in axes:
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.suptitle('LiteEdit-Qwen — Step count sweep (INT4 base model)', y=1.02, fontsize=12)
plt.tight_layout()

fig_path = PATHS.figures / 'step_sweep.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved → {fig_path}')

In [ ]:
# ── Pivot table: paper-friendly view ──────────────────────────────
pivot = df.pivot_table(index='steps', columns='task', values=['lpips', 'latency_s'])
print('\nLPIPS by task × steps (lower = better):')
print(pivot['lpips'].to_string())
print('\nLatency (s) by task × steps:')
print(pivot['latency_s'].to_string())

## What to take from this notebook

1. The right-most plot (quality–speed trade-off) is Figure 2 of your paper.
2. The crossover point — where LPIPS degrades sharply — is different per task.
   That difference *is* the finding.
3. Pick the step count where `bg_replace` still looks acceptable but `local_edit`
   shows visible degradation. That gap motivates the task-aware scheduling section.

**Next:** `03_lora_finetune.ipynb` → train adapters and measure quality recovery.